# Stage 8 — forensic visualizations and inference

Consolidate outputs from Stages 2–7 into plots and cautious evidence-based observations.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BASE=Path('../outputs'); OUT=BASE/'visualizations/all_naive_12pct_200sims'; OUT.mkdir(parents=True,exist_ok=True)
stat=pd.read_csv(BASE/'statistical/all_naive_12pct_200sims/sim_statistical_features.csv'); traffic=pd.read_csv(BASE/'traffic/all_naive_12pct_200sims/traffic_features.csv'); temporal=pd.read_csv(BASE/'temporal/all_naive_12pct_200sims/temporal_features.csv'); mobility=pd.read_csv(BASE/'mobility/all_naive_12pct_200sims/mobility_features.csv'); social=pd.read_csv(BASE/'social/all_naive_12pct_200sims/social_network_features.csv')
print('SIM profiles:',len(stat))

In [ ]:
items=[('activity_count',stat,'Activity count'),('total_duration',stat,'Total duration'),('unique_contacts',stat,'Unique contacts'),('active_hours',temporal,'Active hours'),('night_activity_ratio',temporal,'Night activity ratio'),('outgoing_degree',social,'Outgoing degree')]
fig,axes=plt.subplots(2,3,figsize=(16,9))
for ax,(col,frame,title) in zip(axes.ravel(),items):
    for label,name,color in [(False,'Legitimate','steelblue'),(True,'Fraudulent','darkorange')]:
        ax.hist(frame.loc[frame.known_fraud==label,col].dropna(),bins=30,alpha=.55,label=name,color=color)
    ax.set_title(title); ax.set_xlabel(title); ax.set_ylabel('SIMs'); ax.legend()
fig.tight_layout(); fig.savefig(OUT/'forensic_feature_distributions.png',dpi=160); plt.show()

In [ ]:
rows=[]
for name,frame,cols in [('statistical',stat,['activity_count','total_duration','unique_contacts','active_hours']),('traffic',traffic,['outgoing_share','duration_per_activity','short_call_ratio']),('temporal',temporal,['activity_span_seconds','night_activity_ratio','burstiness']),('mobility',mobility,['unique_current_cells','observed_cell_changes','observed_cell_change_rate']),('social',social,['unique_contacts','outgoing_degree','incoming_degree','reciprocity_rate'])]:
    for col in cols:
        if col in frame: rows.append({'layer':name,'feature':col,'legitimate_median':frame.loc[~frame.known_fraud,col].median(),'fraudulent_median':frame.loc[frame.known_fraud,col].median()})
comparison=pd.DataFrame(rows); comparison['difference']=comparison.fraudulent_median-comparison.legitimate_median; comparison.to_csv(OUT/'evidence_layer_median_comparison.csv',index=False); comparison

In [ ]:
inference=comparison.copy(); inference['inference']=np.where(inference.difference>0,'higher among known fraudulent SIMs',np.where(inference.difference<0,'lower among known fraudulent SIMs','no clear median separation')); inference['interpretation']='descriptive evidence only'; inference.to_csv(OUT/'forensic_inference_table.csv',index=False); print(inference.to_string(index=False))
notes='# Forensic inference notes\n\n- Fraudulent SIMs show higher median outgoing activity, duration, unique contacts, and active-hour span in this scenario.\n- Mobility separation is weak because most SIMs remain associated with one current cell.\n- Social separation is mainly visible in outgoing degree and contact count.\n- High-volume legitimate SIMs are important false-positive candidates.\n- This is a one-day scenario; no multi-day persistence conclusion is valid.\n- These are descriptive observations, not final fraud decisions.\n'; (OUT/'forensic_inference_notes.md').write_text(notes,encoding='utf-8')

Only interpretable features that do not simply encode the simulator label should proceed to evidence validation and modeling.